In [ ]:
import yaml
import os
import logging
from bmc.lake.wekeo import wekeo_lake # Adjust import based on your exact structure

# 1. Setup standard logging for the notebook
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("LakeBuilder")

# 2. Define the Test Recipe
yaml_recipe = """
base_dir: "/storage/niels/bmc"
lake_name: "wekeo_lake"

raw_config:
  keep_raw: false
  spatial:
    use_bbox: true

spatial:
  target_grid: "EEA"
  target_resolution: "100m"
  bbox:
    long_min: 2.54   
    long_max: 5.92   
    lat_min: 50.68   
    lat_max: 51.51   
  # Nests inside spatial and renamed to match the engine's exact keyword expectation
  resampling_strategies:
    continuous: ["average", "max", "min", "rms"]
    discrete: "coverage" 

temporal:
  start_year: 2018
  start_month: 1
  end_year: 2019
  end_month: 12

sources:
  wekeo:
    enabled: true
    query_resolution: "highest" 
    datasets:
      GRA:
        include: true
        productTypes:
          - "Grassland"
      TCF: 
        include: true
        productTypes:
          - "Tree Cover Density"
          - "Forest Type"
"""

# Parse the YAML
recipe = yaml.safe_load(yaml_recipe)

# 3. Emulate the path setup usually done by generate_cube_recipe
base_dir = recipe.get("base_dir", "./test_outputs/")
cube_name = recipe.get("cube_name", "test_cube")

recipe["paths"] = {
    "base_dir": os.path.join(base_dir, cube_name, "lake"),
    "raw_dir": os.path.join(base_dir, cube_name, "raw_downloads")
}

# Ensure the target grid key is formatted correctly for the engine
recipe["spatial"]["target_grid_key"] = f"{recipe['spatial']['target_grid']}_{recipe['spatial']['target_resolution']}"

# REMOVE OR COMMENT OUT THIS LINE:
# recipe["raw_config"] = {"keep_raw": recipe.get("keep_raw", False)}

print("DEBUG RAW CONFIG:", recipe.get('raw_config'))

DEBUG RAW CONFIG: {'keep_raw': False, 'spatial': {'use_bbox': True}}


In [2]:
# Initialize the WEkEO Lake Engine
lake_engine = wekeo_lake()
lake_engine.wekeo_logger = logger 

logger.info("Starting WEkEO Lake Generation Pipeline...")

# PHASE 1 & 2: Fetch and Build
generated_cogs = lake_engine.build_datalake(recipe, logger)

if not generated_cogs:
    logger.warning("No COGs were generated. Check the API query logic or bbox constraints.")
else:
    logger.info(f"Successfully baked {len(generated_cogs)} COGs. Proceeding to Validation...")
    
    # PHASE 3: Validate
    lake_dir = recipe["paths"]["base_dir"]
    is_valid = lake_engine.validate_datalake(lake_dir, logger=logger)
    if is_valid:
        logger.info("Lake validation passed! Generating catalog...")
        
        # PHASE 4: Catalog
        catalog_path = lake_engine.generate_catalog(lake_dir, logger=logger)
        logger.info(f"Pipeline complete! Catalog ready at: {catalog_path}")
    else:
        logger.error("Lake validation failed. See logs for mathematical/ecological violations.")

2026-06-15 10:26:37,742 - INFO - Starting WEkEO Lake Generation Pipeline...
2026-06-15 10:26:37,744 - INFO - Authenticating WEkEO API and building query queue...
2026-06-15 10:26:40,646 - INFO - Successfully loaded pre-compiled inventory: 'wekeo_data_inventory.csv'
2026-06-15 10:26:40,663 - INFO - Queued: Grassland (2018) at 10m
2026-06-15 10:26:40,663 - INFO - Queued: Grassland (2019) at 10m
2026-06-15 10:26:40,681 - INFO - Queued: Tree Cover Density (2018) at 10m
2026-06-15 10:26:40,683 - INFO - Queued: Tree Cover Density (2019) at 10m
2026-06-15 10:26:40,685 - INFO - Queued: Forest Type (2018) at 10m
2026-06-15 10:26:40,685 - INFO - 
Building Lake Layer: Grassland_2018
2026-06-15 10:26:40,689 - INFO - ====EO:EEA:DAT:HRL:GRA====
2026-06-15 10:26:40,690 - INFO - Executing search query...
    {
        "dataset_id": "EO:EEA:DAT:HRL:GRA",
        "bbox": [
            2.54,
            50.68,
            5.92,
            51.51
        ],
        "productType": "Grassland",
        "res

Exporting to Cloud Optimized GeoTIFF: ./test_outputs/micro_lake\lake\GRA\Grassland\coverage\Grassland_2018_EEA_100m_Non_grassland.tif


2026-06-15 10:34:16,050 - WARNING - CPLE_NotSupported in '<Logger LakeBuilder (INFO)>' is an unexpected value for COMPRESS creation option of type string-select.
2026-06-15 10:34:16,050 - WARNING - CPLE_NotSupported in driver COG does not support creation option TILED
2026-06-15 10:34:16,341 - WARNING - CPLE_NotSupported in '<Logger LakeBuilder (INFO)>' is an unexpected value for COMPRESS creation option of type string-select.
2026-06-15 10:34:16,343 - WARNING - CPLE_IllegalArg in COMPRESS=<Logger LakeBuilder (INFO)> value not recognised, ignoring.
2026-06-15 10:34:16,343 - WARNING - CPLE_IllegalArg in COMPRESS_OVERVIEW=<Logger LakeBuilder (INFO)> value not recognised, ignoring.
2026-06-15 10:34:16,698 - INFO - Preparing out-of-core reprojection to EEA_100m...
2026-06-15 10:34:16,700 - INFO - Utilizing 'average' resampling for reprojection.
2026-06-15 10:34:16,701 - INFO - Lazy xarray object detected. Preparing for disk stream...
2026-06-15 10:34:16,703 - INFO - Sanitizing spatial geom

COG successfully generated.


2026-06-15 10:39:39,255 - INFO - Warping to /vsimem/temp_frac_1f617601b39042c4a594dee00cb0cba6.tif (Resampling: average)...
2026-06-15 10:39:45,519 - INFO - Reprojection complete.


...

2026-06-15 11:31:24,160 - INFO - Warping to /vsimem/temp_frac_e6c5c55d234e4135a783ed03e9bab881.tif (Resampling: average)...
2026-06-15 11:31:31,276 - INFO - Reprojection complete.


Exporting to Cloud Optimized GeoTIFF: ./test_outputs/micro_lake\lake\TCF\Forest_Type\coverage\Forest_Type_2018_EEA_100m_Coniferous_forest.tif


2026-06-15 11:31:36,221 - WARNING - CPLE_NotSupported in '<Logger LakeBuilder (INFO)>' is an unexpected value for COMPRESS creation option of type string-select.
2026-06-15 11:31:36,224 - WARNING - CPLE_NotSupported in driver COG does not support creation option TILED
2026-06-15 11:31:36,800 - WARNING - CPLE_NotSupported in '<Logger LakeBuilder (INFO)>' is an unexpected value for COMPRESS creation option of type string-select.
2026-06-15 11:31:36,803 - WARNING - CPLE_IllegalArg in COMPRESS=<Logger LakeBuilder (INFO)> value not recognised, ignoring.
2026-06-15 11:31:36,809 - WARNING - CPLE_IllegalArg in COMPRESS_OVERVIEW=<Logger LakeBuilder (INFO)> value not recognised, ignoring.


COG successfully generated.


2026-06-15 11:31:37,417 - INFO - keep_raw is False. Purging raw data directory: ./test_outputs/micro_lake\raw_downloads
2026-06-15 11:31:37,936 - INFO - Raw data successfully deleted.
2026-06-15 11:31:38,222 - INFO - Successfully baked 8 COGs. Proceeding to Validation...
2026-06-15 11:31:38,259 - INFO - 
=== Initiating Global QA Sweep on: c:\Users\niels\Documents\Repositories\BmC\tutorials\test_outputs\micro_lake\lake ===
2026-06-15 11:31:38,372 - INFO - 
--- Validating: GRA | Grassland | 2018 ---
2026-06-15 11:31:39,094 - INFO -     -> EEA_100m_Grassland Bounds [0,1]: [PASSED]
2026-06-15 11:31:39,488 - INFO -     -> EEA_100m_Non_grassland Bounds [0,1]: [PASSED]
2026-06-15 11:31:40,022 - INFO -     -> Aggregate Sum <= 100%: [PASSED]
2026-06-15 11:31:40,023 - INFO - 
--- Validating: GRA | Grassland | 2019 ---
2026-06-15 11:31:40,472 - INFO -     -> EEA_100m_Grassland Bounds [0,1]: [PASSED]
2026-06-15 11:31:40,747 - INFO -     -> EEA_100m_Non_grassland Bounds [0,1]: [PASSED]
2026-06-15 1

In [9]:
import os
import re
import logging
from pathlib import Path
from typing import Optional
import geopandas as gpd
import pandas as pd
import rioxarray
from shapely.geometry import box
from bmc.utils.logger import log_execution

def fixed_generate_catalog(self, target_dir: str, logger: Optional[logging.Logger] = None) -> str:
    """
    Crawls the finalized COGs and writes a STAC-compliant GeoParquet catalog.
    Extracts specific layer classifications from the filename to enable detailed filtering,
    avoiding redundancy for continuous variables.

    Parameters
    ----------
    target_dir : str
        The root directory of the local data lake containing the finalized COGs.
    logger : logging.Logger, optional
        An optional system logger to record processing milestones and errors.

    Returns
    -------
    str
        The local path to the generated `wekeo_lake_catalog.parquet` database file, 
        or an empty string if no items were inventoried.
    """
    log_execution(logger, f"Generating cloud-native STAC GeoParquet catalog for lake at: {target_dir}", logging.INFO)
    
    base_path = Path(target_dir)
    catalog_records = []
    
    for tif_path in base_path.rglob("*.tif"):
        parts = tif_path.parts
        
        # Anchor to the data lake root folder name to index subdirectories reliably
        if base_path.name in parts:
            base_idx = parts.index(base_path.name)
            if len(parts) - base_idx < 4: 
                continue
            category = parts[base_idx + 1]     # e.g., 'HRL' or 'TCF'
            product = parts[base_idx + 2]      # e.g., 'Tree_Cover_Density'
            aggregation = parts[base_idx + 3]  # e.g., 'average', 'coverage'
        else:
            if len(parts) < 4: 
                continue
            category, product, aggregation = parts[-4], parts[-3], parts[-2]
        
        # Parse timestamp string from filename layout
        year_match = re.search(r'(19\d{2}|20\d{2})', tif_path.name)
        if year_match:
            year_str = year_match.group(1)
            datetime_val = f"{year_str}-01-01T00:00:00Z" 
        else:
            datetime_val = None

        # Extract the specific layer/class from the filename (e.g., 'Non_grassland')
        if "100m_" in tif_path.stem:
            extracted_suffix = tif_path.stem.split("100m_")[-1]
        else:
            # Fallback to grabbing everything after the last underscore
            extracted_suffix = tif_path.stem.split("_")[-1]

        # Prevent redundancy: if the suffix is just the aggregation method (e.g., 'average'),
        # leave layer_class empty, as it's a continuous variable.
        layer_class = None if extracted_suffix == aggregation else extracted_suffix

        # Inspect the physical file to inherit its exact spatial properties
        try:
            with rioxarray.open_rasterio(tif_path) as src:
                epsg_code = src.rio.crs.to_epsg() if src.rio.crs else None
                
                # Transform raster bounds into WGS84 bounding box coordinates
                bounds_wgs84 = src.rio.transform_bounds("EPSG:4326")
                long_min, lat_min, long_max, lat_max = bounds_wgs84
                
                # Generate standard Shapely geometry representing the spatial footprint
                geometry_obj = box(long_min, lat_min, long_max, lat_max)
                
        except Exception as e:
            if logger:
                logger.warning(f"Could not read metadata for footprint extraction on {tif_path.name}: {e}")
            continue

        # Append standard STAC Asset metadata dictionary structure
        catalog_records.append({
            "stac_version": "1.0.0",
            "type": "Feature",
            "id": tif_path.stem,
            "geometry": geometry_obj,
            "bbox": [long_min, lat_min, long_max, lat_max],
            "properties": {
                "datetime": datetime_val,
                "collection": category,
                "variable": product,
                "aggregation": aggregation,
                "layer_class": layer_class,  # Safely added without continuous redundancy
                "proj:epsg": epsg_code,
                "level": category, 
                "year": int(year_str) if year_match else None
            },
            "assets": {
                "data": {
                    "href": str(tif_path.absolute()),
                    "type": "image/tiff; application=geotiff; profile=cloud-optimized"
                }
            }
        })
        
    if not catalog_records:
        log_execution(logger, "No items found to inventory.", logging.WARNING)
        return ""

    # Flatten the records directly onto the root level for clean GeoParquet columns
    flattened_records = []
    for record in catalog_records:
        flat = {
            "stac_version": record["stac_version"],
            "type": record["type"],
            "id": record["id"],
            "geometry": record["geometry"],
            "bbox": record["bbox"],
            "href": record["assets"]["data"]["href"],
            "mime_type": record["assets"]["data"]["type"]
        }
        
        # REMOVED 'prop_' PREFIX: Keeps columns aligned with direct stac-geoparquet conventions
        for prop_k, prop_v in record["properties"].items():
            flat[prop_k] = prop_v
            
        flattened_records.append(flat)

    # Wrap as a standard GeoDataFrame
    gdf = gpd.GeoDataFrame(flattened_records, geometry="geometry", crs="EPSG:4326")
    
    # Export the finalized Parquet database file
    catalog_path = os.path.join(target_dir, "wekeo_lake_catalog.parquet")
    gdf.to_parquet(catalog_path, index=False)
    
    log_execution(logger, f"STAC GeoParquet catalog generated successfully with {len(gdf)} assets at: {catalog_path}", logging.INFO)
    return catalog_path

In [10]:
import types

# Bind the function strictly to your live object instance
lake_engine.generate_catalog = types.MethodType(fixed_generate_catalog, lake_engine)
print("Successfully bound updated method directly to the lake_engine instance!")

Successfully bound updated method directly to the lake_engine instance!


In [11]:
catalog_path = lake_engine.generate_catalog(lake_dir, logger=logger)
logger.info(f"Pipeline complete! Catalog ready at: {catalog_path}")

2026-06-15 12:04:03,561 - INFO - Generating cloud-native STAC GeoParquet catalog for lake at: ./test_outputs/micro_lake\lake
2026-06-15 12:04:04,157 - INFO - STAC GeoParquet catalog generated successfully with 15 assets at: ./test_outputs/micro_lake\lake\wekeo_lake_catalog.parquet
2026-06-15 12:04:04,163 - INFO - Pipeline complete! Catalog ready at: ./test_outputs/micro_lake\lake\wekeo_lake_catalog.parquet


In [12]:
import os
import geopandas as gpd
import pandas as pd

def inspect_stac_catalog(parquet_path: str):
    """
    Loads and profiles a STAC GeoParquet catalog to verify completeness and data integrity.
    """
    if not os.path.exists(parquet_path):
        print(f"Error: Could not find catalog at {parquet_path}")
        return

    print(f"=== Profiling STAC Catalog: {parquet_path} ===")
    
    # Load the GeoParquet file
    gdf = gpd.read_parquet(parquet_path)
    
    # 1. High-Level Summary
    print("\n[1] HIGH-LEVEL SUMMARY")
    print("-" * 30)
    print(f"Total Items (Rows):    {len(gdf)}")
    print(f"Total Fields (Columns): {len(gdf.columns)}")
    print(f"Coordinate Reference:  {gdf.crs}")
    
    # 2. Schema & Missing Values Check
    print("\n[2] SCHEMA & COMPLETENESS")
    print("-" * 30)
    print(f"{'Column Name':<20} | {'Data Type':<15} | {'Missing Values'}")
    print("-" * 60)
    
    null_counts = gdf.isnull().sum()
    for col in gdf.columns:
        dtype_str = str(gdf[col].dtype)
        missing = null_counts[col]
        missing_str = f"{missing} ({missing/len(gdf):.1%})" if missing > 0 else "0 (0.0%)"
        print(f"{col:<20} | {dtype_str:<15} | {missing_str}")

    # 3. Field Values Overview
    print("\n[3] CATEGORICAL INVENTORY OVERVIEW")
    print("-" * 30)
    # These fields correspond to the metadata parsed from your directory/filename logic
    check_fields = ['collection', 'variable', 'aggregation', 'level', 'year', 'proj:epsg', 'mime_type']
    
    for field in check_fields:
        if field in gdf.columns:
            unique_vals = gdf[field].dropna().unique().tolist()
            # Sort if possible for cleaner output
            try: unique_vals.sort()
            except TypeError: pass 
            
            print(f"Unique '{field}':")
            print(f"  -> {unique_vals}")
        else:
            print(f"Warning: Expected field '{field}' is missing from the catalog.")

    # 4. Spatial Extent
    print("\n[4] OVERALL SPATIAL EXTENT")
    print("-" * 30)
    bounds = gdf.total_bounds  # returns [minx, miny, maxx, maxy]
    print(f"Bounding Box (WGS84): [Min Lon: {bounds[0]:.4f}, Min Lat: {bounds[1]:.4f}, Max Lon: {bounds[2]:.4f}, Max Lat: {bounds[3]:.4f}]")

    # 5. Data Sample
    print("\n[5] FIRST RECORD PREVIEW")
    print("-" * 30)
    # Convert the first row to a dict for a clean vertical printout
    sample_record = gdf.iloc[0].to_dict()
    for key, val in sample_record.items():
        print(f"{key:>15}: {val}")

# --- Execution ---
# Replace with the actual path returned by your fixed_generate_catalog function
catalog_filepath = "./test_outputs/micro_lake\lake\wekeo_lake_catalog.parquet"
inspect_stac_catalog(catalog_filepath)

=== Profiling STAC Catalog: ./test_outputs/micro_lake\lake\wekeo_lake_catalog.parquet ===

[1] HIGH-LEVEL SUMMARY
------------------------------
Total Items (Rows):    15
Total Fields (Columns): 15
Coordinate Reference:  {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbrev

In [13]:
gdf = gpd.read_parquet(catalog_filepath)
gdf

,stac_version,type,id,geometry,bbox,href,mime_type,datetime,collection,variable,aggregation,layer_class,proj:epsg,level,year
0,1.0.0,Feature,Grassland_2018_EEA_100m_Grassland,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,GRA,Grassland,coverage,Grassland,3035,GRA,2018
1,1.0.0,Feature,Grassland_2018_EEA_100m_Non_grassland,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,GRA,Grassland,coverage,Non_grassland,3035,GRA,2018
2,1.0.0,Feature,Grassland_2019_EEA_100m_Grassland,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2019-01-01T00:00:00Z,GRA,Grassland,coverage,Grassland,3035,GRA,2019
3,1.0.0,Feature,Grassland_2019_EEA_100m_Non_grassland,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2019-01-01T00:00:00Z,GRA,Grassland,coverage,Non_grassland,3035,GRA,2019
4,1.0.0,Feature,Forest_Type_2018_EEA_100m_Broadleaved_forest,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,TCF,Forest_Type,coverage,Broadleaved_forest,3035,TCF,2018
5,1.0.0,Feature,Forest_Type_2018_EEA_100m_Coniferous_forest,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,TCF,Forest_Type,coverage,Coniferous_forest,3035,TCF,2018
6,1.0.0,Feature,Forest_Type_2018_EEA_100m_Non_forest_areas,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,TCF,Forest_Type,coverage,Non_forest_areas,3035,TCF,2018
7,1.0.0,Feature,Tree_Cover_Density_2018_EEA_100m_average,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,TCF,Tree_Cover_Density,average,NaN,3035,TCF,2018
8,1.0.0,Feature,Tree_Cover_Density_2019_EEA_100m_average,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2019-01-01T00:00:00Z,TCF,Tree_Cover_Density,average,NaN,3035,TCF,2019
9,1.0.0,Feature,Tree_Cover_Density_2018_EEA_100m_max,"POLYGON ((6.91229 49.7831, 6.91229 51.86641, 1...","[1.0179569636385655, 49.78310010267251, 6.9122...",c:\Users\niels\Documents\Repositories\BmC\tuto...,image/tiff; application=geotiff; profile=cloud...,2018-01-01T00:00:00Z,TCF,Tree_Cover_Density,max,NaN,3035,TCF,2018


# spatiotemporal_lake

# Data Lake generation

For some datasources we can not access data in easy way, e.g. we can not subset the data on the providers end. In these cases, the cubing engine must gather all the data itself and construct an offline data lake that creates a compiled version of the data ready to sliced and subsetted on the fly. An example of such a datasource is the WeKEO data provider. In WekEO, not all datasets provide the option to subset based on a bounding box and forces the user to download everything contained within the dataset locally. 

In this notebook we will go through the steps of utilizing the ``spatiotemporal_lake`` class to generate a static data lake from which we can build data cubes on the fly. This lake is stored locally or ideally on the BMD data space. In order to run this we need to go through the following steps

1. Generate a datacatalog that contains all parameters which we need to construct API queries for WekEO
2. Generate the outline of all the datasets and data products we wish to include in the data lake through the use of a recipe
3. Generate a catalog that outlines the lake structure which can then be utilized in the ``spatiotemporal_cube`` as a data access tool. 

## Catalog generation

In order to generate a complete data catalog for the WekEO datasets we need to generate a catalog that can be used to generate queries. Standardly this catalog should be 

In [1]:
from bmc.datasource.wekeo import interface

metadata_schema = dict(zip([key for key in interface.DATASET_MAP.keys()],
                           [interface.get_dataset_variables(interface.DATASET_MAP[key],c)["constraints"][0] for key in interface.DATASET_MAP.keys()]))
metadata_schema

NameError: name 'c' is not defined